[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/13_distill_gkd.ipynb)

# 13 · 知识蒸馏（GKD）：Qwen3-32B → Qwen3-4B-Base

> ⚠️ **只作参考，不参与排名。** 老师 Qwen3-32B 是官方后训练模型；比赛规则（`main` 的 RULES 第 2、3 条）不允许加载其他已后训练模型、也不允许在线蒸馏。
> 这个 notebook 只放在 `submission-yy726` 分支上做研究对比。

结构参考 Unsloth 官方 [Qwen3 (0.6B) Knowledge Distillation](https://github.com/unslothai/notebooks/blob/main/python_scripts/Qwen3_(0.6B)-Knowledge_Distillation.py)：
TRL `GKDTrainer`，Unsloth 把它的损失换成**分块**计算（不在显存里放完整的 `(batch, seq, vocab)` logits）。学生学的是老师在每个 token 上的**概率分布**，比 one-hot 标签信息量大得多。

- `lmbda`：0 = 离线（在数据集给定的答案上对齐老师分布；默认，快）；1 = 在线（学生自己生成、老师打分；要 HF 逐 token 生成，很慢）。
- `beta`：0 = 正向 KL，1 = 反向 KL，0.5 = JSD。
- 学生和老师**必须同词表**（Qwen3 全系列相同；下面有检查）。
- **数据**：默认**短答案**（Dolci-Instruct 科学 / 数学 + MegaScience TextbookReasoning）+ SciQ 选择题格式样本——锁定评测的生成预算是 8192，回答要能收尾；Dolci-Think 长思维链留作选项（见 `experiments/DATASETS.md`）。

**显存**（A100 40GB）：老师 32B 4-bit 约 19GB + 学生 4B 16-bit 约 8GB + LoRA / 激活，约 30–35GB，偏紧。OOM 时把老师换成 `unsloth/Qwen3-14B-unsloth-bnb-4bit`。H100 80GB 可以用 16-bit 老师。

**老师有多强**（Qwen3 技术报告，官方评测、长生成预算）：Qwen3-32B GPQA-Diamond 思考模式 68.4、非思考模式 54.6（Qwen3-4B：55.9 / 41.7）。
同一报告里，8B 学生用**在线蒸馏**（学生采样、老师给逐 token logits）GPQA-D 55.6 → 63.3，比直接 RL（61.3）更好、算力约 1/10。
合规的替代方案：notebook 12 用 OLMo 3 公开的 Dolci-Instruct-DPO（32B vs 0.6B 的 delta learning 偏好对）做 DPO，可以上榜。

Fireworks `distillation_loop` 那种**在线蒸馏**（老师单独部署成服务，对学生的采样逐 token 给 logprob）需要两块卡或推理服务，Colab 单卡放不下，这里不做。

| 模式 | 学生 ← 老师 | 数据 | 时间 |
|---|---|---|---|
| `quick` | Qwen3-0.6B-Base ← Qwen3-1.7B | 约 250 条，≤1024 token，30 步 | T4 约 15 分钟 |
| `a100` | Qwen3-4B-Base ← Qwen3-32B（4-bit） | 约 4500 条，≤4096 token，1 epoch | A100 约 2 小时（估） |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q
sys.path.insert(0, os.path.abspath("scripts"))

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

# GKD：用 Unsloth 官方蒸馏示例测过的 TRL 版本（新版 TRL 把 GKDTrainer 移到了 trl.experimental，Unsloth 的分块损失补丁不一定生效）
!uv pip install --no-deps "trl==0.25.1"

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
# T4（计算能力 7.5）不支持 FlashInfer 采样器（A100 不受影响）
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "unsloth_zoo", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login
login(token = os.environ.get("HF_TOKEN"))
print("W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME = "K1"
MODE     = "a100"                 # "quick" | "a100"
STUDENT  = None                   # None = 用预设；例如 "unsloth/Qwen3-4B-Base"
TEACHER  = None                   # None = 用预设；OOM 时 "unsloth/Qwen3-14B-unsloth-bnb-4bit"
INIT_MODEL = None                 # 可选：从已经 SFT 过的学生开始（例如 notebook 11 的 PUSH_SFT_REPO）
EVAL_BEFORE = True                # 训练前先测一次学生（独立进程，约 5–10 分钟）

LMBDA       = 0.0                 # 0 离线 / 1 在线
BETA        = 0.5                 # 0 正向 KL / 1 反向 KL
TEMPERATURE = 1.0

# 蒸馏数据：每项从一个 HF 数据集抽 n 条（a100 的数量；quick 自动乘 0.05）。见 experiments/DATASETS.md
CHAT_STYLE = "olmo"               # 和 notebook 11 一致：回答以 <|endoftext|> 结束
DISTILL_SOURCES = [
    dict(dataset = "allenai/Dolci-Instruct-SFT-7B", n = 2500, include = ["science", "math", "gsm"],
         exclude = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic"]),
    dict(dataset = "MegaScience/TextbookReasoning", n = 1500),
    # dict(dataset = "allenai/Dolci-Think-SFT-7B", n = 2000, include = ["science", "math"]),   # 思维链（更长）
    # dict(dataset = "open-thoughts/OpenThoughts3-1.2M", n = 2000, include = ["science"]),
]
SCIQ_FORMAT_N = 500               # SciQ 选择题格式样本（"... The answer is (X)."）

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(student="unsloth/Qwen3-0.6B-Base", teacher="unsloth/Qwen3-1.7B", base_model="Qwen/Qwen3-0.6B-Base",
                 teacher_4bit=False, max_seq_len=1024, scale=0.05, steps=30, lr=2e-4, batch=1, accum=4, lora_rank=16,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, eval_gpu_util=0.5,
                 shuffle_buffer=1000),
    a100 = dict(student="unsloth/Qwen3-4B-Base", teacher="unsloth/Qwen3-32B-unsloth-bnb-4bit", base_model="Qwen/Qwen3-4B-Base",
                teacher_4bit=True, max_seq_len=4096, scale=1.0, steps=-1, lr=1e-4, batch=1, accum=8, lora_rank=32,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=4,
                eval_gpu_util=0.5, shuffle_buffer=20000),
)
Q = PRESETS[MODE]
STUDENT, TEACHER = STUDENT or Q["student"], TEACHER or Q["teacher"]
BASE_MODEL, MAX_SEQ_LEN = Q["base_model"], Q["max_seq_len"]
print(EXP_NAME, MODE, "student:", INIT_MODEL or STUDENT, "teacher:", TEACHER, Q)

In [ ]:
import json
from recipe_utils import (TABLE_COLS, run_script, wandb_init, log_eval, print_table, decontam_rows, prepare_sft_sources,
                          sciq_format_rows, config_kwargs, setup_chat, paired_delta)

RUN_NAME = f"{EXP_NAME}-gkd-{STUDENT.split('/')[-1]}-from-{TEACHER.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, dict(
    exp = EXP_NAME, recipe = "gkd", mode = MODE, student = STUDENT, teacher = TEACHER, init_model = INIT_MODEL,
    lmbda = LMBDA, beta = BETA, temperature = TEMPERATURE, sources = DISTILL_SOURCES, sciq_format_n = SCIQ_FORMAT_N, chat_style = CHAT_STYLE,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, reference_only = True, **Q), WANDB_ENTITY)
REPORT = dict(notebook = "13_distill_gkd", exp = EXP_NAME, mode = MODE, student = STUDENT, teacher = TEACHER,
              init_model = INIT_MODEL, lmbda = LMBDA, beta = BETA, temperature = TEMPERATURE,
              sources = DISTILL_SOURCES, q = Q, reference_only = True)
RESULTS, _PER_Q = {}, {}

def quick_eval_subprocess(model_path, tag, gpu_util, samples = None):
    # 独立进程跑 vLLM（scripts/quick_eval.py），结束后显存全部释放；只输出聚合数字
    out = f"outputs/eval_{tag}.json"
    run_script(["scripts/quick_eval.py", "--model", model_path, "--configs", *Q["eval_configs"],
                "--max-tokens", Q["eval_max_tokens"], "--samples", samples or Q["eval_samples"],
                "--gpu-util", gpu_util, "--tag", tag, "--out", out]
               + (["--n", Q["eval_n"]] if Q["eval_n"] else []))
    rep = json.load(open(out))
    RESULTS[tag], _PER_Q[tag] = rep["results"], rep.get("per_question", {})
    log_eval(wandb, RESULTS[tag], tag)

if EVAL_BEFORE:   # 这时还没往 GPU 上加载任何模型；base 不收尾（每个回答写满预算），只采样 1 次
    quick_eval_subprocess(INIT_MODEL or BASE_MODEL, "0_init" if INIT_MODEL else "0_base", 0.85,
                          samples = None if INIT_MODEL else 1)

## 1. 确认 Unsloth 的分块 GKD 损失已接上（否则蒸馏会悄悄退化成普通 SFT）

In [ ]:
import unsloth
from unsloth.models.rl_replacements import RL_FUNCTIONS
_patches = [getattr(f, "__name__", "") for f in RL_FUNCTIONS.get("gkd_trainer", [])]
assert any("compute_loss" in n for n in _patches), (
    f"这个 Unsloth 没有给 GKD 接上分块损失（{_patches}）：Runtime → Restart session 后重跑（让上面的安装生效）")
print("unsloth", unsloth.__version__, "| GKD chunked loss wired:", _patches)

## 2. 加载学生（16-bit LoRA）和老师（冻结）

In [ ]:
import torch, gc
from unsloth import FastLanguageModel

student, tokenizer = FastLanguageModel.from_pretrained(
    INIT_MODEL or STUDENT, max_seq_length = MAX_SEQ_LEN, dtype = None, load_in_4bit = False)
tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
STOP_IDS = setup_chat(tokenizer, student, style = CHAT_STYLE)   # 回答以 eos=<|endoftext|> 结束（照 OLMo 3）
head = student.get_output_embeddings()
assert head.weight.dim() == 2, f"output head is not dense: {type(head.weight).__name__}"

student = FastLanguageModel.get_peft_model(
    student, r = Q["lora_rank"], lora_alpha = 2 * Q["lora_rank"], lora_dropout = 0, bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407)

gc.collect(); torch.cuda.empty_cache()
_free, _total = torch.cuda.mem_get_info(0)
max_memory = {0: max(int(_free - 2.0 * 2**30), 0)}   # 给老师留预算，保留 2GB 余量
print(f"free {_free / 2**30:.1f} of {_total / 2**30:.1f} GiB -> teacher budget {max_memory[0] / 2**30:.1f} GiB")
teacher, teacher_tok = FastLanguageModel.from_pretrained(
    TEACHER, max_seq_length = MAX_SEQ_LEN, dtype = None, load_in_4bit = Q["teacher_4bit"], max_memory = max_memory)
teacher_tok = getattr(teacher_tok, "tokenizer", teacher_tok)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

# 检查：同词表（宽度 + 实际 id）、老师和学生确实不同
s_vocab, t_vocab = student.config.get_text_config().vocab_size, teacher.config.get_text_config().vocab_size
assert s_vocab == t_vocab, f"vocab width mismatch: student {s_vocab} vs teacher {t_vocab}"
probe = "The quick brown fox jumps over 0123456789 lazy dogs, naively. <|im_start|>user\nHi<|im_end|>"
assert tokenizer(probe).input_ids == teacher_tok(probe).input_ids, "tokenizers disagree: GKD cannot compare distributions"
_ids = tokenizer("The capital of France is", return_tensors = "pt").input_ids.to(student.device)
with torch.no_grad():
    _gap = (student(input_ids = _ids).logits.float() - teacher(input_ids = _ids).logits.float()).abs().max().item()
print(f"vocab {s_vocab} ok | max teacher-student logit gap {_gap:.3f}")
print(f"GPU memory after loading both: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## 3. 蒸馏数据

`prepare_sft_sources` 对每个数据源调用 `scripts/prepare_dolci_sft.py`（流式读取、按来源 / domain 过滤、长度 ≤ `MAX_SEQ_LEN`），
再加 SciQ 格式样本，全部过 GPQA 13-gram 去污染。整条放不进 `MAX_SEQ_LEN` 的样本会被丢掉（Unsloth 示例提到超长会导致 prompt 被截空而报错）。

In [ ]:
from datasets import Dataset
from collections import Counter

rows = prepare_sft_sources(DISTILL_SOURCES, STUDENT, MAX_SEQ_LEN, scale = Q["scale"], shuffle_buffer = Q["shuffle_buffer"])
rows += [dict(r, source = "sciq_format", dataset = "allenai/sciq") for r in sciq_format_rows(max(1, int(SCIQ_FORMAT_N * Q["scale"])))]
rows, dropped = decontam_rows(rows, lambda r: {"messages": r["messages"]}, "distill_all")

def _fits(msgs):
    whole = tokenizer.apply_chat_template(msgs, tokenize = True)
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize = True, add_generation_prompt = True)
    whole = whole["input_ids"] if hasattr(whole, "keys") else whole
    prompt = prompt["input_ids"] if hasattr(prompt, "keys") else prompt
    return len(whole) <= MAX_SEQ_LEN and len(prompt) < len(whole)

before = len(rows)
rows = [r for r in rows if _fits(r["messages"])]
import random; random.Random(0).shuffle(rows)
train_ds = Dataset.from_list([{"messages": r["messages"]} for r in rows])
DATA = dict(rows = len(rows), dropped_decontam = dropped, dropped_too_long = before - len(rows),
            by_dataset = Counter(r["dataset"] for r in rows).most_common(),
            tokens = sum(r["n_tokens"] or 0 for r in rows))
REPORT["data"] = DATA
wandb.config.update({"data": DATA}, allow_val_change = True)
print(DATA)

## 4. 训练（`GKDConfig`；不设 fp16/bf16，Unsloth 按架构自动选精度）

In [ ]:
try:
    from trl import GKDConfig, GKDTrainer
except ImportError:                       # 新版 TRL
    from trl.experimental.gkd import GKDConfig, GKDTrainer

config = GKDConfig(**config_kwargs(
    GKDConfig,
    output_dir = "outputs/gkd", per_device_train_batch_size = Q["batch"], gradient_accumulation_steps = Q["accum"],
    warmup_steps = 10, max_steps = Q["steps"], num_train_epochs = 1, learning_rate = Q["lr"], logging_steps = 5,
    optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear", seed = 3407,
    report_to = REPORT_TO, run_name = RUN_NAME, save_strategy = "no",
    max_length = MAX_SEQ_LEN, lmbda = LMBDA, beta = BETA, temperature = TEMPERATURE, max_new_tokens = 512))
trainer = GKDTrainer(model = student, teacher_model = teacher, args = config, train_dataset = train_ds,
                     processing_class = tokenizer)
assert type(trainer.args).__name__ == "GKDConfig", f"config was downcast to {type(trainer.args).__name__}"
assert trainer.args.lmbda == LMBDA and trainer.args.beta == BETA

before = {n: p.detach().clone() for n, p in student.named_parameters() if p.requires_grad}
torch.cuda.reset_peak_memory_stats()
result = trainer.train()
changed = sum(1 for n, p in student.named_parameters() if p.requires_grad and not torch.equal(p.detach(), before[n]))
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
GKD = dict(steps = trainer.state.global_step, minutes = round(result.metrics["train_runtime"] / 60, 1),
           loss_first = losses[0] if losses else None, loss_last = losses[-1] if losses else None,
           adapters_changed = f"{changed}/{len(before)}", peak_mem_gb = round(torch.cuda.max_memory_allocated() / 2**30, 1))
REPORT["gkd"] = GKD
print(GKD)
assert changed > 0, "no adapter changed: the student did not learn"

## 5. 合并学生 → 独立进程评测（先释放老师的显存）

In [ ]:
del trainer, teacher, before
gc.collect(); torch.cuda.empty_cache()
student.save_pretrained_merged("outputs/merged_student", tokenizer, save_method = "merged_16bit")
quick_eval_subprocess("outputs/merged_student", "1_gkd", Q["eval_gpu_util"])

## 6. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
# 训练前后的配对比较（同一批题，后 − 前，单位：分；|z| < 2 = 看不出差别）
COMPARE = {}
_tags = list(_PER_Q)
for _a, _b in zip(_tags, _tags[1:]):
    COMPARE[f"{_a} -> {_b}"] = {cfg: paired_delta(_PER_Q[_a][cfg], _PER_Q[_b][cfg])
                                for cfg in _PER_Q[_b] if cfg in _PER_Q[_a]}
print("compare:", COMPARE)

rows = print_table(RESULTS)
REPORT.update(quick_eval = RESULTS, compare = COMPARE, versions = VERSIONS, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB,
              wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 7.（可选）推到 Hub

蒸馏结果只作参考、不上排行榜；如果想用 `01` 跑锁定评测做对比，可以推到私有仓库后再评测。

In [ ]:
PUSH = False
HUB_REPO = "YOUR_HF_USERNAME/" + STUDENT.split("/")[-1].lower() + f"-{EXP_NAME.lower()}-gkd"
if PUSH:
    student.push_to_hub_merged(HUB_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{HUB_REPO}")